# Import statements

In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, roc_auc_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Load, concatenate, and clean data

In [13]:
spotify_high_pop = pd.read_csv("/Users/donyabehroozi/Documents/gsb545/GSB-545/Project Files/high_popularity_spotify_data.csv")
spotify_high_pop.head()

,energy,tempo,danceability,playlist_genre,loudness,liveness,valence,track_artist,time_signature,speechiness,...,instrumentalness,track_album_id,mode,key,duration_ms,acousticness,id,playlist_subgenre,type,playlist_id
0,0.592,157.969,0.521,pop,-7.777,0.122,0.535,"Lady Gaga, Bruno Mars",3,0.0304,...,0.0000,10FLjwfpbxLmW8c25Xyc2N,0,6,251668,0.3080,2plbrEY59IikOBgBGLjaoe,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M
1,0.507,104.978,0.747,pop,-10.171,0.117,0.438,Billie Eilish,4,0.0358,...,0.0608,7aJuG4TFXa2hmE4z1yxc3n,1,2,210373,0.2000,6dOtVTDdiauQNBQEDOtlAB,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M
2,0.808,108.548,0.554,pop,-4.169,0.159,0.372,Gracie Abrams,4,0.0368,...,0.0000,0hBRqPYPXhr1RkTDG3n4Mk,1,1,166300,0.2140,7ne4VBA60CxGM75vw0EYad,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M
3,0.910,112.966,0.670,pop,-4.070,0.304,0.786,Sabrina Carpenter,4,0.0634,...,0.0000,4B4Elma4nNDUyl6D5PvQkj,0,0,157280,0.0939,1d7Ptw3qYcfpdLNL5REhtJ,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M
4,0.783,149.027,0.777,pop,-4.477,0.355,0.939,"ROSÉ, Bruno Mars",4,0.2600,...,0.0000,2IYQwwgxgOIn7t3iF6ufFD,0,0,169917,0.0283,5vNRhkKd0yEAg8suGBpjeY,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M


In [14]:
spotify_low_pop = pd.read_csv('/Users/donyabehroozi/Documents/gsb545/GSB-545/Project Files/low_popularity_spotify_data.csv')
spotify_low_pop.head()

,time_signature,track_popularity,speechiness,danceability,playlist_name,track_artist,duration_ms,energy,playlist_genre,playlist_subgenre,...,track_album_id,playlist_id,track_id,valence,key,tempo,loudness,acousticness,liveness,track_album_name
0,4.0,23,0.0393,0.636,Rock Classics,Creedence Clearwater Revival,138053.0,0.746,rock,classic,...,4A8gFwqd9jTtnsNwUu3OQx,37i9dQZF1DWXRqgorJj26U,5e6x5YRnMJIKvYpZxLqdpH,0.432,0.0,132.310,-3.785,0.0648,0.1730,The Long Road Home - The Ultimate John Fogerty...
1,4.0,53,0.0317,0.572,Rock Classics,Van Halen,241600.0,0.835,rock,classic,...,2c965LEDRNrXXCeBOAAwns,37i9dQZF1DWXRqgorJj26U,5FqYA8KfiwsQvyBI4IamnY,0.795,0.0,129.981,-6.219,0.1710,0.0702,The Collection
2,4.0,55,0.0454,0.591,Rock Classics,Stevie Nicks,329413.0,0.804,rock,classic,...,3S404OgKoVQSJ3xXrDVlp8,37i9dQZF1DWXRqgorJj26U,5LNiqEqpDc8TuqPy79kDBu,0.658,0.0,111.457,-7.299,0.3270,0.0818,Bella Donna (Deluxe Edition)
3,4.0,64,0.1010,0.443,Jazz Classics,"Ella Fitzgerald, Louis Armstrong",185160.0,0.104,jazz,classic,...,1y5KGkUKO0NG32MhIIagCA,37i9dQZF1DXbITWG1ZJKYt,78MI7mu1LV1k4IA2HzKmHe,0.394,0.0,76.474,-17.042,0.9130,0.1910,"Love, Ella"
4,4.0,62,0.0298,0.685,Jazz Classics,Galt MacDermot,205720.0,0.472,jazz,classic,...,6f4b9wVTkKAf096k4XG6x5,37i9dQZF1DXbITWG1ZJKYt,6MN6yRVriszuyAVlyF8ndB,0.475,9.0,80.487,-9.691,0.7850,0.2240,Shapes of Rhythm/Woman Is Sweeter


In [15]:
#make binary popular variable (1 = popular track, 0 = not a popular track)
spotify_high_pop["popular"] = 1
spotify_low_pop["popular"] = 0

#concatenate high and low popularity datasets
spotify_full = pd.concat([spotify_high_pop, spotify_low_pop], axis=0).reset_index(drop=True)

In [16]:
#drop NAs
spotify_full = spotify_full.dropna()

#fix release year
def parse_release_year(date_str):
    if pd.isna(date_str):
        return None
    date_str = str(date_str).strip()
    try:
        return pd.to_datetime(date_str).year
    except:
        return None

spotify_full['release_year'] = spotify_full['track_album_release_date'].apply(parse_release_year)

#drop irrelevant columns
drop_cols = ['track_artist', 'track_href', 'uri', 'track_album_name',
             'playlist_name', 'analysis_url', 'track_id', 'track_name',
             'track_album_id', 'id', 'type', 'playlist_id',
             'track_album_release_date',
             'track_popularity']

spotify_full = spotify_full.drop(columns=drop_cols)

In [17]:
#collapse playlist genres and subgenres
genre_map = {
    'electronic': 'electronic', 'ambient': 'electronic', 'lofi': 'electronic',
    'gaming': 'electronic', 'disco': 'electronic',
    'pop': 'pop', 'indie': 'pop', 'k-pop': 'pop', 'j-pop': 'pop',
    'cantopop': 'pop', 'mandopop': 'pop',
    'hip-hop': 'hip-hop_rnb', 'r&b': 'hip-hop_rnb', 'soul': 'hip-hop_rnb',
    'funk': 'hip-hop_rnb',
    'latin': 'latin_world', 'world': 'latin_world', 'arabic': 'latin_world',
    'brazilian': 'latin_world', 'afrobeats': 'latin_world', 'turkish': 'latin_world',
    'indian': 'latin_world', 'korean': 'latin_world', 'soca': 'latin_world',
    'reggae': 'latin_world',
    'rock': 'rock_metal', 'metal': 'rock_metal', 'punk': 'rock_metal',
    'jazz': 'jazz_blues', 'blues': 'jazz_blues', 'gospel': 'jazz_blues',
    'classical': 'classical_folk', 'folk': 'classical_folk',
    'wellness': 'wellness', 'country': 'wellness'
}

subgenre_map = {
    'chill': 'chill', 'lofi': 'chill', 'meditative': 'chill',
    'yoga': 'chill', 'soft': 'chill', 'bedroom': 'chill', 'smooth': 'chill',
    'modern': 'modern', 'mainstream': 'modern', 'pop': 'modern',
    'feel-good': 'modern', 'essential': 'modern',
    'classic': 'classic', 'throwback': 'classic', '80s': 'classic',
    '90s': 'classic', 'retro': 'classic',
    'hip-hop': 'urban', 'trap': 'urban', 'gangster': 'urban',
    'drill': 'urban', 'grime': 'urban',
    'deep house': 'electronic', 'techno': 'electronic', 'hardstyle': 'electronic',
    'future': 'electronic', 'vaporwave': 'electronic', 'future bass': 'electronic',
    'afro house': 'electronic',
    'reggaeton': 'latin', 'tropical': 'latin', 'afro-latin': 'latin',
    'cumbia': 'latin', 'samba': 'latin', 'forró': 'latin', 'carnival': 'latin',
    'french': 'world', 'scandi': 'world', 'nordic': 'world',
    'african': 'world', 'global': 'world', 'chinese': 'world',
    'japanese': 'world', 'nigerian': 'world', 'desi': 'world',
    'bhangra': 'world', 'bollywood': 'world', 'amapiano': 'world',
    'gqom': 'world', 'throat singing': 'world', 'australian': 'world',
    'celtic': 'world', 'irish': 'world', 'indigenous': 'world',
    'klezmer': 'world', 'jewish': 'world', 'tango': 'world',
    'cajun': 'world', 'southern': 'world', 'american': 'world',
    'classical': 'classical', 'neo-classical': 'classical', 'choral': 'classical',
    'cinematic': 'classical', 'academic': 'classical', 'soundtracks': 'classical',
    'noir': 'classical', 'drama': 'classical',
    'alternative': 'alternative', 'indie': 'alternative', 'pop punk': 'alternative',
    'death': 'alternative', 'heavy': 'alternative', 'experimental': 'alternative',
    'post-rock': 'alternative', 'avant-garde': 'alternative',
    'funk': 'other', 'melodic': 'other', 'workout': 'other',
    'anime': 'other', 'italo': 'other', 'delta': 'other',
    'fusion': 'other', 'latin': 'other', 'spanish': 'other',
    'afrobeats': 'other'
}

spotify_full['playlist_genre'] = spotify_full['playlist_genre'].map(genre_map)
spotify_full['playlist_subgenre'] = spotify_full['playlist_subgenre'].map(subgenre_map)

# Data Preprocessing

In [ ]:
#define quantitative variables
quant_vars = [
    'energy', 'tempo', 'danceability', 'loudness',
    'liveness', 'valence', 'time_signature',
    'speechiness', 'instrumentalness',
    'mode', 'key', 'duration_ms', 'acousticness', 'release_year'
]
 
#define categorical variables
cat_vars = ['playlist_genre', 'playlist_subgenre']

#define target variaable
y = spotify_full['popular']
 
#define predictors
X = pd.concat([
    spotify_full[quant_vars],
    pd.get_dummies(spotify_full[cat_vars], drop_first=True).astype(int)
], axis=1)
 
# train + test + validation split
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=321)
 
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, stratify=y_train_full, test_size=0.2, random_state=321)

# SMOTE

In [19]:
#SMOTE applied to training data only
sm = SMOTE(random_state=321)
X_train_smote, y_train_smote = sm.fit_resample(X_train, y_train)
 
print("Class distribution before SMOTE:")
print(y_train.value_counts(normalize=True))
print("\nClass distribution after SMOTE:")
print(pd.Series(y_train_smote).value_counts(normalize=True))
print(f"\nTraining set size before SMOTE: {len(X_train)}")
print(f"Training set size after SMOTE:  {len(X_train_smote)}")

Class distribution before SMOTE:
popular
0    0.651133
1    0.348867
Name: proportion, dtype: float64

Class distribution after SMOTE:
popular
0    0.5
1    0.5
Name: proportion, dtype: float64

Training set size before SMOTE: 3090
Training set size after SMOTE:  4024


In [20]:
#scaled variable data preprocessing for logisitc regression and MLP
scaler = MinMaxScaler()
X_train_smote_scaled = X_train_smote.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()
X_train_smote_scaled[quant_vars] = scaler.fit_transform(X_train_smote[quant_vars])
X_val_scaled[quant_vars] = scaler.transform(X_val[quant_vars])
X_test_scaled[quant_vars] = scaler.transform(X_test[quant_vars])

# SMOTE Baseline Models

In [ ]:
#no class weights since SMOTE handles class imbalance
rf_model = RandomForestClassifier(random_state=321, n_jobs=-1)
lgbm_model = LGBMClassifier(random_state=321, n_jobs=-1, verbose=-1)
lr_model = LogisticRegression(max_iter=1000, random_state=321)
nb_model = GaussianNB()

In [22]:
#define cross validation method
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=321)

In [23]:
#random forest model evaluation
cv_scores_rf = cross_val_score(
    ImbPipeline([('smote', SMOTE(random_state=321)), ('model', RandomForestClassifier(random_state=321, n_jobs=-1))]),
    X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
print(cv_scores_rf)
print(cv_scores_rf.mean())
print(cv_scores_rf.std())
 
rf_model.fit(X_train_smote, y_train_smote)
 
y_proba = rf_model.predict_proba(X_val)[:, 1]
print("Random Forest SMOTE Baseline ROC AUC (Validation):", roc_auc_score(y_val, y_proba))
print(classification_report(y_val, rf_model.predict(X_val)))

[0.89640487 0.8882913
 0.89390432 0.89811936
 0.88480053]
0.8923040772630968
0.005011991036201568
Random Forest SMOTE Baseline ROC AUC (Validation): 0.8936013548339592
              precision    recall  f1-score   support

           0       0.86      0.90      0.88       503
           1       0.80      0.72      0.76       270

    accuracy                           0.84       773
   macro avg       0.83      0.81      0.82       773
weighted avg       0.84      0.84      0.84       773



In [24]:
#random forest evaluation (Test data)
y_proba = rf_model.predict_proba(X_test)[:, 1]
print("Random Forest SMOTE Baseline ROC AUC (Test):", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, rf_model.predict(X_test)))

Random Forest SMOTE Baseline ROC AUC (Test): 0.913946587537092
              precision    recall  f1-score   support

           0       0.87      0.91      0.89       629
           1       0.81      0.74      0.78       337

    accuracy                           0.85       966
   macro avg       0.84      0.83      0.83       966
weighted avg       0.85      0.85      0.85       966



In [29]:
#lgbm SMOTE evaluation
cv_scores_lgbm = cross_val_score(
    ImbPipeline([('smote', SMOTE(random_state=321)), ('model', LGBMClassifier(random_state=321, n_jobs=-1, verbose=-1))]),
    X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
print(cv_scores_lgbm)
print(cv_scores_lgbm.mean())
print(cv_scores_lgbm.std())
 
lgbm_model.fit(X_train_smote, y_train_smote)
 
y_proba = lgbm_model.predict_proba(X_val)[:, 1]
print("LightGBM SMOTE Baseline ROC AUC (Validation):", roc_auc_score(y_val, y_proba))
print(classification_report(y_val, lgbm_model.predict(X_val)))

[0.89403312 0.90285648
 0.89927101 0.90186222
 0.89076608]
0.8977577813628898
0.004645212695989481
LightGBM SMOTE Baseline ROC AUC (Validation): 0.9099329946248436
              precision    recall  f1-score   support

           0       0.89      0.91      0.90       503
           1       0.82      0.79      0.80       270

    accuracy                           0.87       773
   macro avg       0.85      0.85      0.85       773
weighted avg       0.86      0.87      0.86       773



In [30]:
#lgbm test evaluation
y_proba = lgbm_model.predict_proba(X_test)[:, 1]
print("LightGBM SMOTE Baseline ROC AUC (Test):", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, lgbm_model.predict(X_test)))

LightGBM SMOTE Baseline ROC AUC (Test): 0.9262193769961269
              precision    recall  f1-score   support

           0       0.87      0.92      0.89       629
           1       0.83      0.75      0.79       337

    accuracy                           0.86       966
   macro avg       0.85      0.83      0.84       966
weighted avg       0.86      0.86      0.86       966



In [33]:
#logistic regression SMOTE evaluation
cv_scores_lr = cross_val_score(
    ImbPipeline([('smote', SMOTE(random_state=321)), ('model', LogisticRegression(max_iter=5000, random_state=321, solver='saga'))]),
    X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
print(cv_scores_lr)
print(cv_scores_lr.mean())
print(cv_scores_lr.std())
 
lr_model.fit(X_train_smote_scaled, y_train_smote)
 
y_proba = lr_model.predict_proba(X_val_scaled)[:, 1]
print("Logistic Regression SMOTE Baseline ROC AUC (Validation):", roc_auc_score(y_val, y_proba))
print(classification_report(y_val, lr_model.predict(X_val_scaled)))

[0.5832939  0.59949218
 0.64608094 0.58885549
 0.63139165]
0.609822832589765
0.024617068183701782
Logistic Regression SMOTE Baseline ROC AUC (Validation): 0.8235623297253516
              precision    recall  f1-score   support

           0       0.80      0.86      0.83       503
           1       0.70      0.61      0.65       270

    accuracy                           0.77       773
   macro avg       0.75      0.73      0.74       773
weighted avg       0.77      0.77      0.77       773



In [34]:
#logistic regression test evaluation
y_proba = lr_model.predict_proba(X_test_scaled)[:, 1]
print("Logistic Regression SMOTE Baseline ROC AUC (Test):", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, lr_model.predict(X_test_scaled)))

Logistic Regression SMOTE Baseline ROC AUC (Test): 0.8534553929038131
              precision    recall  f1-score   support

           0       0.83      0.88      0.86       629
           1       0.75      0.67      0.71       337

    accuracy                           0.81       966
   macro avg       0.79      0.77      0.78       966
weighted avg       0.80      0.81      0.80       966



In [35]:
#naive bayes SMOTE evaluation
cv_scores_nb = cross_val_score(
    ImbPipeline([('smote', SMOTE(random_state=321)), ('model', GaussianNB())]),
    X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
print(cv_scores_nb)
print(cv_scores_nb.mean())
print(cv_scores_nb.std())
 
nb_model.fit(X_train_smote, y_train_smote)
 
y_proba = nb_model.predict_proba(X_val)[:, 1]
print("Naive Bayes SMOTE Baseline ROC AUC (Validation):", roc_auc_score(y_val, y_proba))
print(classification_report(y_val, nb_model.predict(X_val)))

[0.7380749  0.69938254
 0.73456214 0.72085752
 0.74076377]
0.7267281751923715
0.015292912150084048
Naive Bayes SMOTE Baseline ROC AUC (Validation): 0.7277888226198365
              precision    recall  f1-score   support

           0       0.88      0.42      0.56       503
           1       0.45      0.89      0.60       270

    accuracy                           0.58       773
   macro avg       0.66      0.65      0.58       773
weighted avg       0.73      0.58      0.58       773



In [36]:
#naive bayes test evaluation
y_proba = nb_model.predict_proba(X_test)[:, 1]
print("Naive Bayes SMOTE Baseline ROC AUC (Test):", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, nb_model.predict(X_test)))

Naive Bayes SMOTE Baseline ROC AUC (Test): 0.7743887193180263
              precision    recall  f1-score   support

           0       0.89      0.42      0.57       629
           1       0.45      0.91      0.61       337

    accuracy                           0.59       966
   macro avg       0.67      0.66      0.59       966
weighted avg       0.74      0.59      0.58       966



In [37]:
#MLP SMOTE evaluation
tf.random.set_seed(321)
 
input_dim = X_train_smote_scaled.shape[1]
inputs = keras.Input(shape=(input_dim,))
x = layers.Dense(64, activation='relu')(inputs)
x = layers.Dense(32, activation='relu')(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs=inputs, outputs=outputs, name="spotify_popularity_mlp_smote")
 
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(name='auc')]
)
 
history = model.fit(
    X_train_smote_scaled, y_train_smote,
    epochs=20,
    batch_size=32,
    validation_data=(X_val_scaled, y_val),
    verbose=1
)
 
y_proba = model.predict(X_val_scaled).flatten()
y_pred = (y_proba > 0.5).astype(int)
print("Keras MLP SMOTE Baseline ROC AUC (Validation):", roc_auc_score(y_val, y_proba))
print(classification_report(y_val, y_pred, zero_division=0))

Epoch 1/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - auc: 0.8178 - loss: 0.5631 - val_auc: 0.8080 - val_loss: 0.5000
Epoch 2/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 483us/step - auc: 0.8997 - loss: 0.4071 - val_auc: 0.8364 - val_loss: 0.4712
Epoch 3/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 532us/step - auc: 0.9199 - loss: 0.3621 - val_auc: 0.8492 - val_loss: 0.4528
Epoch 4/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 582us/step - auc: 0.9283 - loss: 0.3389 - val_auc: 0.8578 - val_loss: 0.4417
Epoch 5/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 547us/step - auc: 0.9333 - loss: 0.3247 - val_auc: 0.8634 - val_loss: 0.4346
Epoch 6/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 506us/step - auc: 0.9367 - loss: 0.3151 - val_auc: 0.8675 - val_loss: 0.4310
Epoch 7/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 486us/step - auc: 0.9392 - loss: 0.3079 - val_auc: 0.8708 - val_loss: 0.4272
Epoch 8/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 476us/step - auc: 0.9412 - loss: 0.3022 - val_auc: 0.8736 - val_loss: 0.4250
Epoch 9/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s

In [38]:
#MLP test evaluation
y_pred_prob = model.predict(X_test_scaled).flatten()
y_pred = (y_pred_prob > 0.5).astype(int)
roc_auc = roc_auc_score(y_test, y_pred_prob)
print("\nTest Set Evaluation:")
print(f"MLP SMOTE Baseline Test ROC AUC Score: {roc_auc:.4f}")
print(classification_report(y_test, y_pred, target_names=['Not Popular', 'Popular']))

31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 387us/step

Test Set Evaluation:
MLP SMOTE Baseline Test ROC AUC Score: 0.8834
              precision    recall  f1-score   support

 Not Popular       0.80      0.94      0.87       629
     Popular       0.85      0.57      0.68       337

    accuracy                           0.81       966
   macro avg       0.82      0.76      0.77       966
weighted avg       0.82      0.81      0.80       966

